# TaskIntent Grammar Strategy

This notebook implements a research-only generation strategy named `task_intent_grammar`.

Pipeline:

`prompt -> TaskIntent -> broad grammar context -> structural rules -> RobotDesignIR -> benchmark`

The existing packaged `grammar` strategy wraps `grammar_loop.build_structural_rules()`. That is useful as a baseline, but it still carries product-loop assumptions. This notebook shows how to write a strategy directly against the research interfaces, using `TaskIntent` as the task-side source of truth and leaving embodiment choices to emerge from the rules and materialized IR.

The strategy is registered at notebook runtime only. It does not edit `apps/`, `packages/pipeline/`, or the permanent strategy registry.

LLM-backed cells use `packages.research.local_chat_models.make_structured_llm(...)`, not direct `ChatOpenAI`. To run through local Codex instead of OpenAI, set `RESEARCH_LLM_PROVIDER=codex` before starting the notebook. For Claude Code, set `RESEARCH_LLM_PROVIDER=claude-code` and `RESEARCH_LLM_MODEL=claude-sonnet-4-5`. To use a custom persistent wrapper around Claude Code or Codex, also set `RESEARCH_LLM_COMMAND` and `RESEARCH_LLM_PROTOCOL=jsonl`.

## 1. Imports and Repo Setup

A notebook may be launched from different working directories. This cell finds the repo root, adds it to `sys.path`, and imports the research and pipeline interfaces the strategy needs.

In [ ]:
from __future__ import annotations

import dataclasses
import hashlib
import json
import re
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "packages" / "research").exists() and (candidate / "packages" / "pipeline").exists():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pydantic import TypeAdapter

from packages.pipeline.grammar_graph import (
    compile_structural_rules,
    fetch_grammar_from_db,
    fetch_rules_from_db,
)
from packages.pipeline.ir.design_ir import (
    ActuatorSlot,
    Collision,
    Geometry,
    Inertial,
    JointIR,
    JointLimits,
    JointType,
    LinkIR,
    RobotDesignIR,
    Vector3,
    Visual,
)
from packages.pipeline.schemas import TaskIntent
from packages.research.benchmark.harness import evaluate_designs
from packages.research.experiment.runner import ExperimentRunner
from packages.research.strategy.protocol import GenerationStrategy, StrategyConfig
from packages.research.strategy.registry import list_strategies, register_strategy

TASK_INTENT_ADAPTER: TypeAdapter[TaskIntent] = TypeAdapter(TaskIntent)

print("Repo root:", ROOT)
print("Existing strategies:", list_strategies())

## 2. Trace Records

Research strategies should make the loop inspectable. A trace record is deliberately small: phase, message, and JSON-friendly data. You can later persist the same records in SQLite or compare them across runs.

In [ ]:
@dataclass(frozen=True)
class StrategyTrace:
    phase: str
    message: str
    data: dict[str, Any] = field(default_factory=dict)


def _jsonable(value: Any) -> Any:
    if dataclasses.is_dataclass(value):
        return dataclasses.asdict(value)
    if isinstance(value, tuple):
        return [_jsonable(v) for v in value]
    if isinstance(value, list):
        return [_jsonable(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    return value


def emit_trace(trace: list[StrategyTrace], phase: str, message: str, **data: Any) -> None:
    trace.append(StrategyTrace(phase=phase, message=message, data=_jsonable(data)))


def print_trace(trace: list[StrategyTrace]) -> None:
    for i, item in enumerate(trace, start=1):
        print(f"{i:02d}. {item.phase}: {item.message}")
        if item.data:
            print(json.dumps(item.data, indent=2, default=str)[:1600])

## 3. TaskIntent Parsing

`TaskIntent` is already the repo's structured task-side schema. This parser keeps the fallback simple and deterministic. It extracts only task-side fields such as terrain, payload needs, sensing needs, spatial constraints, stability needs, and failure modes.

The parser does not create an intermediate embodiment label. If a prompt contains a word like "quadruped", it stays in `task_goal` as user text rather than becoming a control variable.

In [ ]:
def parse_task_intent(prompt: str, config: StrategyConfig, trace: list[StrategyTrace]) -> TaskIntent:
    use_llm = bool(config.extra.get("use_llm_intent", False))
    if use_llm:
        try:
            from packages.research.local_chat_models import make_structured_llm

            llm = make_structured_llm(config.model_id, TaskIntent)
            if llm is None:
                raise RuntimeError("no configured research chat model")
            system = (
                "Convert the user request into TaskIntent JSON. "
                "Use only task-side fields. Do not introduce an embodiment label."
            )
            result = llm.invoke(f"{system}\n\nUser request:\n{prompt}")
            intent = result if isinstance(result, TaskIntent) else TASK_INTENT_ADAPTER.validate_python(result)
            emit_trace(trace, "parse_intent", "LLM parser returned TaskIntent", intent=TASK_INTENT_ADAPTER.dump_python(intent))
            return intent
        except Exception as exc:
            emit_trace(trace, "parse_intent", "LLM parser unavailable; using deterministic fallback", error=str(exc))

    lower = prompt.lower()

    def hits(words: tuple[str, ...]) -> tuple[str, ...]:
        return tuple(word for word in words if word in lower)

    terrain = hits(("stairs", "rocky", "sand", "mud", "ice", "pipes", "rubble", "grass", "gravel"))
    contact = hits(("climb", "crawl", "push", "pull", "grip", "brace", "step over"))
    payload = hits(("carry", "payload", "tool", "sample", "box", "pack"))
    manipulation = hits(("grasp", "turn valve", "open", "assemble", "pick", "place"))
    spatial = hits(("tight", "narrow", "compact", "small", "low profile", "confined"))
    stability = hits(("stable", "balance", "upright", "vibration", "slip"))
    sensing = hits(("inspect", "camera", "sensor", "lidar", "see", "scan"))

    avoid: list[str] = []
    for pattern in (r"avoid [a-z ]+", r"must not [a-z ]+", r"do not [a-z ]+"):
        avoid.extend(match.group(0).strip() for match in re.finditer(pattern, lower))

    intent = TaskIntent(
        task_goal=prompt.strip(),
        terrain=terrain,
        contact_requirements=contact,
        payload_requirements=payload,
        manipulation_requirements=manipulation,
        spatial_constraints=spatial,
        stability_requirements=stability,
        sensing_requirements=sensing,
        failure_modes_to_avoid=tuple(avoid),
        success_criteria=("satisfy task-side requirements",),
    )
    emit_trace(trace, "parse_intent", "deterministic TaskIntent fallback", intent=TASK_INTENT_ADAPTER.dump_python(intent))
    return intent

## 4. Broad Grammar Context

This strategy fetches the full grammar context it can access. It does not ask for family-filtered examples. If Supabase is unavailable, the notebook falls back to a tiny teaching vocabulary so the strategy remains understandable offline.

In [ ]:
@dataclass(frozen=True)
class GrammarContext:
    vocabulary: tuple[tuple[str, str], ...]
    rule_memory: dict[str, str]
    source: str
    errors: tuple[str, ...] = ()

    @property
    def known_nodes(self) -> set[str]:
        return {short for short, _ in self.vocabulary}


LOCAL_VOCABULARY: tuple[tuple[str, str], ...] = (
    ("S", "Start symbol"),
    ("BODY_CHAIN", "Repeatable body scaffold"),
    ("BODY_SEG", "Body segment"),
    ("LOW_PROFILE_BODY", "Compact low-clearance body"),
    ("CONTACT_PAD", "Contact or traction pad"),
    ("STABILIZER", "Passive stabilizing support"),
    ("SENSOR_MAST", "Sensor support structure"),
    ("PAYLOAD_BAY", "Payload support bay"),
    ("ARM", "Manipulation appendage"),
    ("GRIPPER", "End-effector module"),
)

LOCAL_RULE_MEMORY: dict[str, str] = {
    "S -> BODY_CHAIN": "Start with a body scaffold.",
    "BODY_CHAIN -> BODY_SEG CONTACT_PAD": "Attach contact support to a body segment.",
    "BODY_CHAIN -> LOW_PROFILE_BODY STABILIZER": "Use compact body with passive stability support.",
    "BODY_SEG -> SENSOR_MAST": "Mount sensing on a stable body segment.",
    "BODY_SEG -> PAYLOAD_BAY": "Reserve a structural bay for payload support.",
    "ARM -> GRIPPER": "Terminate manipulation appendage with an end effector.",
}


def load_grammar_context(trace: list[StrategyTrace]) -> GrammarContext:
    try:
        vocabulary = tuple(fetch_grammar_from_db(limit=200, active_only=True))
        rule_memory = fetch_rules_from_db(limit=80, active_only=True)
        if not vocabulary:
            raise RuntimeError("GrammarNodes returned no vocabulary rows")
        emit_trace(
            trace,
            "load_grammar_context",
            "loaded broad grammar context from DB",
            vocabulary_count=len(vocabulary),
            rule_count=len(rule_memory),
        )
        return GrammarContext(vocabulary=vocabulary, rule_memory=rule_memory, source="db")
    except Exception as exc:
        emit_trace(
            trace,
            "load_grammar_context",
            "DB grammar context unavailable; using local teaching context",
            error=str(exc),
            vocabulary_count=len(LOCAL_VOCABULARY),
            rule_count=len(LOCAL_RULE_MEMORY),
        )
        return GrammarContext(
            vocabulary=LOCAL_VOCABULARY,
            rule_memory=dict(LOCAL_RULE_MEMORY),
            source="local",
            errors=(str(exc),),
        )

## 5. Rule Builder

The deterministic builder is intentionally simple. It creates compile-shaped structural rules from task-side fields, not from an embodiment label. A future strategy can swap this cell for an LLM rule builder while keeping the same strategy interface and trace shape.

In [ ]:
def choose_node(preferred: tuple[str, ...], known: set[str], fallback: str | None = None) -> str | None:
    for node in preferred:
        if node in known:
            return node
    if fallback and fallback in known:
        return fallback
    return None


def build_rules_from_task_intent(
    intent: TaskIntent,
    context: GrammarContext,
    config: StrategyConfig,
    trace: list[StrategyTrace],
) -> dict[str, list[str]]:
    known = context.known_nodes
    start = choose_node(("S",), known)
    body_chain = choose_node(("BODY_CHAIN", "BODY", "BODY_SEG"), known)
    body = choose_node(("LOW_PROFILE_BODY", "BODY_SEG", "BODY"), known)

    if start is None or body_chain is None or body is None:
        emit_trace(trace, "build_rules", "insufficient vocabulary for structural rules", known_nodes=sorted(known))
        return {}

    rules: dict[str, list[str]] = {start: [body_chain]}
    body_children: list[str] = [body]

    def add_if_needed(field_values: tuple[str, ...], candidate_nodes: tuple[str, ...]) -> None:
        if not field_values:
            return
        node = choose_node(candidate_nodes, known)
        if node and node not in body_children:
            body_children.append(node)

    add_if_needed(intent.contact_requirements or intent.terrain, ("CONTACT_PAD", "STABILIZER", "BODY_SEG"))
    add_if_needed(intent.stability_requirements or intent.spatial_constraints, ("STABILIZER", "LOW_PROFILE_BODY", "BODY_SEG"))
    add_if_needed(intent.sensing_requirements, ("SENSOR_MAST", "BODY_SEG"))
    add_if_needed(intent.payload_requirements, ("PAYLOAD_BAY", "BODY_SEG"))
    add_if_needed(intent.manipulation_requirements, ("ARM", "GRIPPER", "BODY_SEG"))

    rules[body_chain] = body_children

    if "ARM" in body_children and "GRIPPER" in known:
        rules["ARM"] = ["GRIPPER"]

    emit_trace(
        trace,
        "build_rules",
        "built structural rules from TaskIntent fields",
        rules=rules,
        context_source=context.source,
    )
    return rules

## 6. Compile Feasibility Check

Compile validity is recorded as feasibility. It is not an alignment score. If the DB-backed compiler is unavailable, the strategy records that fact and continues to materialization so the notebook stays usable offline.

In [ ]:
def local_rule_check(rules: dict[str, list[str]], known_nodes: set[str]) -> bool:
    if not rules:
        return False
    return all(lhs in known_nodes and all(rhs in known_nodes for rhs in rhs_nodes) for lhs, rhs_nodes in rules.items())


def check_compile_feasibility(
    rules: dict[str, list[str]],
    context: GrammarContext,
    trace: list[StrategyTrace],
) -> dict[str, Any]:
    local_ok = local_rule_check(rules, context.known_nodes)
    try:
        db_ok = bool(compile_structural_rules(rules))
        result = {"compile_safe": db_ok, "local_rule_check": local_ok, "reason": "DB-backed compiler completed"}
    except Exception as exc:
        result = {
            "compile_safe": False,
            "local_rule_check": local_ok,
            "reason": f"DB-backed compiler unavailable: {exc}",
        }

    emit_trace(trace, "compile_rules", "recorded feasibility check", **result)
    return result

## 7. TaskIntent Materializer

This notebook-local materializer converts structural rules into `RobotDesignIR`. It receives only task intent JSON and graph-node context. It does not accept or render an intermediate embodiment label.

In [ ]:
@dataclass(frozen=True)
class NodeContext:
    node_name: str
    role: str
    neighbors: tuple[str, ...]
    depth: int
    is_root: bool


def build_node_contexts(rules: dict[str, list[str]]) -> list[NodeContext]:
    if not rules:
        return []

    children = {child for rhs in rules.values() for child in rhs}
    roots = [lhs for lhs in rules if lhs not in children] or [next(iter(rules))]
    contexts: dict[str, NodeContext] = {}
    queue: list[tuple[str, tuple[str, ...], int, bool]] = [(roots[0], (), 0, True)]

    while queue:
        node, neighbors, depth, is_root = queue.pop(0)
        if node in contexts:
            continue
        role = "root" if is_root else ("intermediate" if node in rules else "leaf")
        contexts[node] = NodeContext(node_name=node, role=role, neighbors=neighbors, depth=depth, is_root=is_root)
        for child in rules.get(node, []):
            queue.append((child, (node,), depth + 1, False))

    return list(contexts.values())


def deterministic_jitter(seed: int, node: str, slot: int, scale: float = 0.18) -> float:
    digest = hashlib.sha256(f"{seed}:{node}:{slot}".encode()).hexdigest()
    return (int(digest[:8], 16) / 0xFFFFFFFF - 0.5) * scale


class TaskIntentMaterializer:
    def materialize(
        self,
        raw_output: dict[str, Any],
        prompt: str,
        config: StrategyConfig,
    ) -> list[RobotDesignIR]:
        rules = raw_output.get("structural_rules") or {}
        intent = raw_output.get("task_intent")
        if not isinstance(intent, TaskIntent):
            intent = TaskIntent(task_goal=prompt)

        contexts = build_node_contexts(rules)
        if not contexts:
            return []

        links: list[LinkIR] = []
        joints: list[JointIR] = []

        for index, ctx in enumerate(contexts):
            links.append(self._make_link(ctx, intent, config.seed + index))
            if not ctx.is_root and ctx.neighbors:
                joints.append(self._make_joint(ctx, ctx.neighbors[0], config.seed + index))

        ir = RobotDesignIR(
            name=f"task_intent_grammar_{config.seed}",
            links=links,
            joints=joints,
            version="notebook-0.1.0",
            source_candidate_id=f"task_intent_grammar_seed_{config.seed}",
        )
        return [ir]

    def _make_link(self, ctx: NodeContext, intent: TaskIntent, seed: int) -> LinkIR:
        node = ctx.node_name.lower()
        compact = bool(intent.spatial_constraints)
        payload = bool(intent.payload_requirements)

        if ctx.is_root:
            base = (0.32, 0.18, 0.10) if not compact else (0.24, 0.13, 0.08)
            mass = 4.5 + (1.0 if payload else 0.0)
            geometry = Geometry(type="box", size=tuple(round(v * (1 + deterministic_jitter(seed, ctx.node_name, i)), 4) for i, v in enumerate(base)))
        elif "sensor" in node:
            geometry = Geometry(type="cylinder", size=(0.025, 0.18))
            mass = 0.35
        elif "payload" in node:
            geometry = Geometry(type="box", size=(0.18, 0.12, 0.08))
            mass = 1.2
        elif "pad" in node or "stabilizer" in node:
            geometry = Geometry(type="box", size=(0.10, 0.06, 0.025))
            mass = 0.45
        elif "gripper" in node:
            geometry = Geometry(type="capsule", size=(0.025, 0.10))
            mass = 0.30
        else:
            scale = max(0.45, 1.0 - 0.12 * ctx.depth)
            geometry = Geometry(type="capsule", size=(round(0.035 * scale, 4), round(0.18 * scale, 4)))
            mass = round(0.9 * scale, 3)

        inertial = Inertial(mass=round(mass * (1 + deterministic_jitter(seed, ctx.node_name, 9)), 3))
        return LinkIR(
            name=ctx.node_name,
            inertial=inertial,
            visual=Visual(geometry=geometry, rgba=(0.35, 0.42, 0.50, 1.0)),
            collision=Collision(geometry=geometry),
        )

    def _make_joint(self, ctx: NodeContext, parent: str, seed: int) -> JointIR:
        node = ctx.node_name.lower()
        fixed = "sensor" in node or "payload" in node or "pad" in node
        joint_type = JointType.FIXED if fixed else JointType.REVOLUTE
        actuator = None if fixed else ActuatorSlot(actuator_type="position", max_torque=8.0 + ctx.depth)
        limits = None if fixed else JointLimits(lower=-1.047, upper=1.047, effort=1.0 + ctx.depth)
        return JointIR(
            name=f"joint_{parent}_to_{ctx.node_name}",
            joint_type=joint_type,
            parent_link=parent,
            child_link=ctx.node_name,
            axis=Vector3(0.0, 0.0, 1.0),
            limits=limits,
            actuator=actuator,
            damping=0.1,
        )

## 8. The Strategy Class

This is the complete `GenerationStrategy`. The loop lives inside `generate()`, which is the key design rule for research strategies in this repo.

In [ ]:
class TaskIntentGrammarStrategy:
    def __init__(self, materializer: TaskIntentMaterializer | None = None) -> None:
        self._materializer = materializer or TaskIntentMaterializer()
        self.last_trace: list[StrategyTrace] = []

    @property
    def name(self) -> str:
        return "task_intent_grammar"

    @property
    def version(self) -> str:
        return "notebook-0.1.0"

    def generate(self, prompt: str, config: StrategyConfig) -> list[RobotDesignIR]:
        trace: list[StrategyTrace] = []
        self.last_trace = trace

        intent = parse_task_intent(prompt, config, trace)
        context = load_grammar_context(trace)
        rules = build_rules_from_task_intent(intent, context, config, trace)
        compile_result = check_compile_feasibility(rules, context, trace)

        raw_output = {
            "task_intent": intent,
            "structural_rules": rules,
            "compile_result": compile_result,
            "grammar_context_source": context.source,
        }

        designs: list[RobotDesignIR] = []
        for i in range(config.max_candidates):
            variant_config = StrategyConfig(
                seed=config.seed + i,
                model_id=config.model_id,
                max_candidates=1,
                prompt_registry=config.prompt_registry,
                extra=config.extra,
            )
            designs.extend(self._materializer.materialize(raw_output, prompt, variant_config))

        emit_trace(
            trace,
            "materialize_ir",
            "materialized RobotDesignIR variants",
            design_count=len(designs),
            design_names=[ir.name for ir in designs],
            validation_errors={ir.name: ir.validate() for ir in designs},
        )
        return designs[: config.max_candidates]


assert isinstance(TaskIntentGrammarStrategy(), GenerationStrategy)
register_strategy("task_intent_grammar", TaskIntentGrammarStrategy)
print("Registered strategies:", list_strategies())

## 9. Inspect One Strategy Run Directly

Run the strategy directly first. This makes the trace available on the instance, which is better for learning than going through the experiment runner immediately.

In [ ]:
prompt = "robot that can inspect pipes, stay stable in tight spaces, and avoid getting stuck"
strategy = TaskIntentGrammarStrategy()
config = StrategyConfig(seed=42, model_id="offline", max_candidates=3)
designs = strategy.generate(prompt, config)

print_trace(strategy.last_trace)
print("\nDesign summary")
for ir in designs:
    print(f"- {ir.name}: links={len(ir.links)} joints={len(ir.joints)} errors={ir.validate()}")

## 10. Benchmark the Designs

The benchmark harness reports feasibility and structural metrics. These are useful covariates for alignment research, but they are not task-alignment labels by themselves.

In [ ]:
report = evaluate_designs(designs)
print(f"total_designs={report.total_designs}")
print(f"compile_rate={report.compile_rate:.0%}")
print(f"stability_rate={report.stability_rate:.0%}")
print(f"mean_screening_score={report.mean_screening_score:.3f}")
print(f"link_count_entropy={report.link_count_entropy:.2f}")
print(f"joint_count_entropy={report.joint_count_entropy:.2f}")

## 11. Optional ExperimentRunner Demo

Set `RUN_EXPERIMENT_DEMO = True` to write a run into the local research SQLite store. It is off by default so importing or validating the notebook does not create experiment state.

In [ ]:
RUN_EXPERIMENT_DEMO = False

if RUN_EXPERIMENT_DEMO:
    runner = ExperimentRunner()
    result = runner.run(
        prompt="robot that can inspect pipes, stay stable in tight spaces, and avoid getting stuck",
        strategy_name="task_intent_grammar",
        experiment_name="task-intent-notebook-demo",
        seed=42,
        max_candidates=3,
    )
    print(f"Run ID: {result.run_id}")
    print(f"Designs: {len(result.designs)}")
    print(f"Error: {result.error}")
    if result.metrics_report:
        print(f"Compile rate: {result.metrics_report.compile_rate:.0%}")
        print(f"Mean screening score: {result.metrics_report.mean_screening_score:.3f}")

## 12. Optional Baseline Comparison

The packaged `grammar` strategy is still useful as a baseline. Turn this on when you want to compare the research-native loop against the existing wrapper around `grammar_loop.build_structural_rules()`.

In [ ]:
RUN_BASELINE_COMPARISON = False

if RUN_BASELINE_COMPARISON:
    runner = ExperimentRunner()
    baseline = runner.run(
        prompt=prompt,
        strategy_name="grammar",
        experiment_name="task-intent-baseline-comparison",
        seed=42,
        max_candidates=3,
    )
    candidate = runner.run(
        prompt=prompt,
        strategy_name="task_intent_grammar",
        experiment_name="task-intent-baseline-comparison",
        seed=42,
        max_candidates=3,
    )
    print("baseline", baseline.run_id, baseline.error)
    print("candidate", candidate.run_id, candidate.error)

## 13. Turning This Notebook Into a Package Strategy

When the strategy is mature enough to leave the notebook:

1. Move `StrategyTrace`, `GrammarContext`, `parse_task_intent`, `load_grammar_context`, `build_rules_from_task_intent`, `check_compile_feasibility`, `TaskIntentMaterializer`, and `TaskIntentGrammarStrategy` into `packages/research/strategy/task_intent_grammar_strategy.py`.
2. Add `TaskIntentGrammarStrategy` to the builtin registry in `packages/research/strategy/registry.py`.
3. Move the deterministic teaching vocabulary into tests or fixtures if the packaged strategy should remain runnable without DB access.
4. Add unit tests in `tests/test_research_core.py` for registration, no intermediate embodiment label in materialization context, trace records, and fallback behavior.
5. Keep alignment labels outside this strategy. Put benchmark cases and judge outputs under `packages/research/benchmark/` so generation and evaluation stay separable.